# E22 Speed Lab rev 2b — vLLM 0.32.0 (Qwen3.8-27B on 4x L4), target 1500+ tok/s

FP8 block (gated vs the FP8 target-only oracle): fp8_s16_triton_full (FULL CUDA graphs with MTP), fp8_s16_ssm_bf16 (fused CUDA GDN verify kernel), fp8_s16_local_argmax (use_local_argmax_reduction), fp8_s16_ssm_fp16 (the fp16 control, same run). INT4 block (gated vs the INT4 target-only oracle int4_target_only_noprefix): int4_s20_profile (torch profiler, per-category kernel time per step), int4_s20 (baseline), int4_s16, MTP off / K1 at 16 and 20 streams (int4_s16_mtp0, int4_s20_mtp0, int4_s16_mtp1, int4_s20_mtp1), int4_s20_local_argmax, int4_s20_triton_full, int4_s20_triton_fdec, int4_s20_nccl2 / nccl4 / nccl8, int4_s20_ring (NCCL_ALGO=Ring), int4_s20_pmu64 (--prefix-match-unit 64), int4_s20_pmu64_sharedckpt (+ --enable-mamba-shared-prefix-checkpoint), int4_s20_pmu16_sharedckpt, int4_s24, int4_s20_b8192, int4_s24_triton_full_nccl8 (combination). Budget 600 min.

## Base: the Exp 14 VMS matrix

The vLLM twin of the EMS (SGLang) notebook: every vLLM configuration worth testing for 600-800 tok/s, each its own
server launch. Reference = the S1-60K engine (LB 34.58): FP8 block-128, fp8 KV, MTP 3, FLASHINFER, prefix caching,
16 seqs. Arms: the oracle (FP8 target-only, prefix caching off); MTP 2 / 4; 20 / 24 seqs; SSM state fp16; prefix
caching off; --linear-backend triton (Marlin = the auto default on SM 8.9); INT4 (RedHat compressed-tensors) + MTP at 16 and 20 after a static MTP
check; last, 2 x TP2 (data parallel 2 x tensor parallel 2) from a reserved slot.
Each arm: a greedy correctness gate vs the oracle's own A/A (rev 2; INT4 arms diagnostic: other weights) with a
degenerate-output count, then 8k and 40k steady windows measured from /metrics. Report `q27x/vms/vms_report.md`, bundle `Q27X_VMS_BUNDLE.zip`.

**Inputs:** competition data; the FP8 repack and the RedHat INT4 models; the vLLM 0.32.0 wheelhouse
(`foysalemonshanto/vllm-0-32-0-cu13-wheelhouse`); the vLLM 0.19 wheelhouse (only the kernel's tokenizer pins); the
SGLang wheelhouse is NOT needed; the payload dataset (E14 modules).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "VMS").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600,   # E15P rev 2: = S60 6H
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "EVAL14":    {"source": "eval", "task_list": ["135a2760", "136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "3dc255db", "4c3d4a41", "4c416de3", "4c7dc4dd", "62593bfd", "64efde09", "65b59efc", "6e4f6532"],
                  "time_budget_sec": 2.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 step 3.5: GPU smoke, 1 fast task (2ba387bc: 5,355 thinking tokens in E10 ORDER2), 2 branches, 15 min solving
    "SMOKE1":    {"source": "eval", "task_list": ["2ba387bc"], "time_budget_sec": 50 * 60, "admit_frac": 0.90,
                  "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60,
                  "admit_margin_sec": 2 * 60, "solve_window_sec": 15 * 60},
    # E12 step 4: the E11-05 EVAL30 tasks (same seed sample), 4 h
    "EVAL30":    {"source": "eval", "n_tasks": 30, "time_budget_sec": 4.0 * 3600,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 rev 4 (user 2026-10-20): EVAL6 = 6 tasks that S49/R5/DIRECT solved and E12 EVAL30 did not (accuracy test)
    "EVAL6":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},   # audits: the 100k cap is soft; 12 x ~120k = 1.44 M at 368 tok/s = 65 min + prefill
    # E12 rev 5 (user 2026-10-21): the same 6 tasks with every validated fix (A/B against rev 4 EVAL6: 3/6, 4/8)
    "EVAL6R5":   {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},
    # E12 rev 5 (user 2026-10-21): EVAL6R5 + 6 informative EVAL30 tasks: 4 Duck rev 1 solved (247ef758 Duck-only on
    # query 1 and first-exact-wrong; bf45cf4b 9.5k; f931b4a8 38.8k; 65b59efc 67k) + 2 mid ones it missed (9385bd28 and
    # cbebaa4b, ledger solve rate 0.25). 24 branches / 6 live; 150 min solving, 185 min budget
    "EVAL12R5":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    # E12 rev 5: the E12 EVAL30 tasks (same seed sample) with every fix (A/B against rev 1 EVAL30 7/48 and Poetiq R5 16/48).
    # Capacity: rev 4 EVAL6 measured about 10.4 min per task at 6 live -> 30 tasks about 5.2 h of solving
    "EVAL30R5":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # E12 rev 6 (user 2026-10-21): the EVAL12R5 tasks with rev 6 (A/B: rev 5 xhigh EVAL6R5 5/8, low EVAL12R5 10/19)
    "EVAL12R6":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    "EVAL30R6":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # gate 4b (user 2026-10-21): the EVAL12R6 tasks at the hidden run's compute share. CELL 6 sets the solve window to
    # n_tasks x (hidden solve seconds / 240), the hidden solve end taken from RERUN_ALL and the start measured in this run
    "EVAL12R6G4B": {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                    "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                    "time_budget_sec": 80 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                    "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                    "hidden_equiv": True},
    # R6-X1 (user 2026-10-21): target-only vs DFlash2 parity on 4 tasks (no solving): DFlash2 twice (control), then the
    # same server without --speculative-*, the same requests at T 0, one at a time, thinking on and off
    "PARITYR6":  {"source": "eval", "task_list": ["136b0064", "bf45cf4b", "247ef758", "b5ca7ac4"],
                  "time_budget_sec": 75 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "parity": True},
    # E14-A (user 2026-10-23: one notebook; plan Notes/13): engine measurements (A1), a 25-minute Duck run with the
    # adopted levers on the G4B tasks (raw reasoning on), more engine arms (A2). Every phase has its own deadline;
    # whatever does not fit before the hard stop is skipped and reported, never squeezed.
    "E14A":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 200 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 25 * 60, "e14": True},
    # E14S (Notes/30, 31; user 2026-10-24: a short commit): a valid version + a GPU smoke of the submission arm (DFlash
    # width 6 + CUDA graphs 1..13) at 12 streams, no measurements. 36 min, not 30: R2 was READY at ~12.5 min (install
    # 3 + staging 2 + first launch 502 s) and the SGLang give-up is admit - 5 min (30 min -> 13 min, 36 min -> 19 min)
    "E14S":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 36 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 8 * 60},
    # E14S EVAL120 gate 4b (user 2026-10-24): every evaluation task (120 / 172 inputs) with the hidden run's compute per
    # task: CELL 6 sets the window to 120 x (hidden solve seconds / 240) = about 5.8 h; the coverage of the hidden run
    # (stage-1 starts, branch 1 starts, pass1 / pass2) on eval difficulty. 6.5 h budget = start + window + margins
    # E15 (Notes/35): engine arms (no E14A measurements: E14_DO_MEASURE stays False, so the first launch = the submission
    # arm W6 + graphs13 = the reference), then a 15-min Duck run on the G4B tasks with the adopted arm. 180 min budget (about 15 launches)
    "E15":       {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 180 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 15 * 60, "e15": True},
    # E15B commit (user 2026-10-25, Exp 15 Notes/08): 20 eval tasks with BOTH attempts finishing = the 16 G4B tasks + the
    # first 4 by id of the tasks solved in 1-2 of the 3 EVAL120 G4B runs (2026-10-24). 40 attempts on 16 streams = 3 waves;
    # rev 3f / 3g: waves 16 x a1, then 4 x a1 + 12 x a2, then the last 4 x a2 (4 running = faster). 45k + 30k at 22 tok/s:
    # worst ~ 83 min -> window 92, budget 115; 40k + 35k: worst ~ 74 min -> window 85, budget 110 (READY took 12-13 min;
    # the solve deadline = hard stop - 180 s still clamps it)
    "E15B":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b",
                                                  "2ba387bc", "1818057f", "45a5af55", "9aaea919",
                                                  "16de56c4", "1ae2feb7", "221dfab4", "269e22fb"],
                  "time_budget_sec": 128 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 105 * 60},
    # E15B EVAL120 gate 4b: every evaluation task at the hidden run's compute per task (as E14SEVAL120G4B)
    "E15BEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # rev 3i (user 2026-10-25): all 120 eval tasks, exactly a 6-h notebook; the solve window = the hidden run's per-task
    # share (120 x (hidden solve end - start) / 240 ~ 5.8 h), clamped by this run's hard stop - 180 s (~5.7 h)
    "E15BEVAL120R3I": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # E15P (user 2026-10-25: Poetiq on the Duck engine): all 120 evaluation tasks, a 6-h notebook like the E11-05 S60
    # EVAL120 6H run (3af4660a) and the Duck R3K EVAL120 runs; the solve runs to the drain (12 min before the hard stop)
    "E15PEVAL120": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90,
                    "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                    "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # VMS (user 2026-10-25): the vLLM 0.32.0 engine-config matrix; the 12 G4B tasks only feed real ARC prompts; 260 min (rev 2d)
    "VMS":       {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 600 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "vms": True},   # rev 2d: 260 min
    "E14SEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # E11 rev 3: 60,000 = S1-60K (LB 34.58); user floor 60k   # S49: LB test of 49,152 (S1: 60,000; R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
E11_THINK_FLOOR = 60000      # E11 rev 3 (user 2026-10-18): never below 60k (budget and admission guard)
assert THINK_BUDGET >= E11_THINK_FLOOR, (THINK_BUDGET, E11_THINK_FLOOR)
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
# ---- E11-05: engine = SGLang 0.5.21 + RedHat INT4 (W4A16 g128) + DFlash2 drafter = the E11-04 W1 launch (2026-10-18).
# MODEL_PATH (FP8) stays: its tokenizer + chat template render every prompt exactly as in S49, and it is the vLLM fallback.
ENGINE_BACKEND = os.environ.get("E11_ENGINE", "sglang").strip().lower()       # sglang (vLLM FP8 fallback) | vllm (= S49)
E11_ALLOW_VLLM_FALLBACK = os.environ.get("E11_VLLM_FALLBACK", "1") == "1"
SGL_MODEL_PATH = Path(os.environ.get("E11_INT4", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"))
SGL_DRAFT_PATH = Path(os.environ.get("E11_DFLASH", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-dflash2/pytorch/bf16/1"))
SGL_WHEELHOUSE = Path(os.environ.get("E11_SGL_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse"))


def _e11_discover(path, marker, pred):
    """The given path if it holds `marker`, else the first /kaggle/input folder whose `marker` json satisfies pred."""
    if (path / marker).is_file():
        return path
    root = Path("/kaggle/input")
    for p in (sorted(root.rglob(marker)) if root.exists() else []):
        try:
            if pred(p.parent, _load_json(p)):
                return p.parent
        except Exception:
            continue
    return path


SGL_MODEL_PATH = _e11_discover(SGL_MODEL_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and (c.get("quantization_config") or {}).get("quant_method") == "compressed-tensors")
SGL_DRAFT_PATH = _e11_discover(SGL_DRAFT_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and c.get("architectures") == ["DFlash2DraftModel"])
SGL_WHEELHOUSE = _e11_discover(SGL_WHEELHOUSE, "BUILD_INFO.json", lambda d, c: isinstance(c, dict)
                               and c.get("sglang") == "0.5.21" and (d / "requirements.lock").is_file())
SGL_CONTEXT_LEN = 155648            # = the E11-04 W1 launch; every request stays inside MAX_MODEL_LEN (phase1_cap)
SGL_MAX_RUNNING = 16                # E15B (Notes/02 Rev 2c): 16 real streams; was 12 (E11-04 W1: 12 really running (mamba cache 56 / 4 state slots = 14 >= 12)
SGL_SERVER_MAX_RUNNING = 17         # E15B: client 16 + 1 (Notes/30 nodelay crash: a 13th on buffers sized 12); rev 3c: server admission limit = client 12 + 1. SGLang 0.5.21 counts the row of a request in
                                    # chunked prefill twice (scheduler.py:3971-3976) and latches batch_is_full (3846-3848) until a
                                    # running request ends -> EVAL30 ran 11 + 1 queued (2,358 of 2,509 decode lines). Upstream fix
                                    # PR #44401 (after 0.5.21); +1 is the documented workaround (PR #30295). CUDA graphs stay at 12.
SGL_LAUNCH_TIMEOUT_SEC = 30 * 60    # first launch includes JIT + CUDA graphs: 567 s in E11-04
SGL_RELAUNCH_TIMEOUT_SEC = 20 * 60  # a diagnosed relaunch (E11-04: about 75 s once the JIT cache exists)
SGL_GIVE_UP_AT = min(NOTEBOOK_T0 + (240 if RERUN else 75) * 60, ADMIT_DEADLINE - 5 * 60)   # E15B B5: RERUN 4 h   # no SGLang launch after this -> vLLM fallback
SGL_STREAM_IDLE_SEC = 1800          # a socket read silent for longer closes that request as 'error'
# rev 2 (GPT audit 2026-10-18, verified): capacity gate from the server log, fail-closed on missing evidence
SGL_MIN_RUNNING = 8                 # below this the SGLang engine is refused (-> S49 vLLM)
SGL_MIN_KV = 600_000                # KV tokens; E11-04 W1 had 881,336
SGL_KV_PER_REQUEST = 56_000         # E15P rev 2 (F6, agent B H1): 852,152 // 56k = 15 Poetiq streams. Poetiq units reach the 60k cap as a cohort: the E11-06 decode-line peak 0.73 of 866,744 = 51.9k per request x 16 = 0.97 of the E15B pool; simulated SGLang retractions at 16: 2 / 8 six-hour and 2 / 4 twelve-hour runs, at 15: 0 / 28 (-1 % tokens). The client KV gate cannot stop running units from growing. A smaller pool adapts (784k -> 14, 700k -> 12). Server mrr / graphs stay 17. E15B was 50k: the MEAN attempt context ~ 50k (attempt 1 at 45k ends at ~57-59k p90, max ~64k; attempt 2 less; Auditor 4); was 70k (KV // 70k = 12). concurrency <= KV // this (881,336 -> 12); a long Poetiq unit is prompt + 49k + 8k
SGL_MIN_FREE_GB = 1.0               # GPU headroom after start-up (E11-04 W1: 2.22 GB)
# rev 2: drain earlier. Aborted long prefixes were not cached in E11-04 (30k prefix: 0 cached, 24.5 s), so 12 phase-2
# continuations re-prefill up to ~0.7M tokens; 12 min instead of 8 min before the hard stop (SGLang only)
SGL_DRAIN_MARGIN_SEC = 12 * 60     # E15P: = S60 (S1-60K); E11-06 used 8 min
# rev 2: KV gate active on SGLang (S49's floor 12 = CONCURRENCY 12 made it inert): from 10 running, admit only below 85 %
SGL_KV_GATE_MIN_ACTIVE = 10
SGL_KV_ADMIT_MAX = 0.85

CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e11"] = {k: (str(globals()[k]) if isinstance(globals()[k], Path) else globals()[k]) for k in (
    "ENGINE_BACKEND", "E11_ALLOW_VLLM_FALLBACK", "SGL_MODEL_PATH", "SGL_DRAFT_PATH", "SGL_WHEELHOUSE", "SGL_CONTEXT_LEN",
    "SGL_MAX_RUNNING", "SGL_SERVER_MAX_RUNNING", "SGL_LAUNCH_TIMEOUT_SEC", "SGL_RELAUNCH_TIMEOUT_SEC", "SGL_STREAM_IDLE_SEC", "SGL_MIN_RUNNING",
    "SGL_MIN_KV", "SGL_KV_PER_REQUEST", "SGL_MIN_FREE_GB", "SGL_DRAIN_MARGIN_SEC", "SGL_KV_GATE_MIN_ACTIVE", "SGL_KV_ADMIT_MAX")}
CONFIG["e11"]["give_up_after_sec"] = round(SGL_GIVE_UP_AT - NOTEBOOK_T0)
print("E11 engine:", json.dumps(CONFIG["e11"]))
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','5'))   # E15P: = S60 (S1-60K)
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','1')=='1'   # E15P: = S60 (S1-60K)
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}



# ---- E15P (user 2026-10-25): the Poetiq solver of E11-05 S60 (S1-60K, LB 34.58) on the E15B engine. No Duck: the Duck
# package, solver and report cells are not in this notebook. The engine cell reads one Duck name (the mamba-track test
# flag); it stays off as in every E15B run.
E12_MAMBA_TRACK_64 = False
# rev 2 (Notes/19): F2 mid-run server restarts (12-stream config, only with >= 45 min before the admission deadline),
# F5 sandbox self-test attempts (it runs while the model copy reads with 24 threads)
E15P_MAX_RESTARTS = int(os.environ.get("E15P_MAX_RESTARTS", 2))
E15P_RESTART_MIN_LEFT_SEC = float(os.environ.get("E15P_RESTART_MIN_LEFT_SEC", 45 * 60))
E15P_SELFTEST_TRIES = int(os.environ.get("E15P_SELFTEST_TRIES", 3))
CONFIG["e15p"] = {"solver": "poetiq-s60", "streams": SGL_MAX_RUNNING, "server_max_running": SGL_SERVER_MAX_RUNNING,
                  "kv_per_request": SGL_KV_PER_REQUEST, "drain_margin_sec": SGL_DRAIN_MARGIN_SEC, "rev": 2,
                  "max_restarts": E15P_MAX_RESTARTS, "restart_min_left_sec": E15P_RESTART_MIN_LEFT_SEC,
                  "engine_source": "6baa9361", "poetiq_source": "3af4660a"}
print("E15P:", json.dumps(CONFIG["e15p"]), "| poetiq", json.dumps(CONFIG["poetiq"]))


# ---- VMS (user 2026-10-25): the vLLM 0.32.0 engine matrix. vLLM reads the Kaggle Models directly (no /dev/shm staging:
# two checkpoints, FP8 + INT4, would not both fit); the rerun of this notebook only writes the cell-0 fallback.
VMS_ON = bool(P.get("vms")) and not RERUN
os.environ["E14_LOADPATH"] = "0"
VLLM_WHEELHOUSE = Path(os.environ.get("VMS_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"))
CONFIG["vms"] = {"on": VMS_ON, "wheelhouse": str(VLLM_WHEELHOUSE)}
print("VMS:", json.dumps(CONFIG["vms"]))


In [ ]:
# ==================================================================== CELL 1b — E14: first-line facts, phase record, staging
# Experiment 14 (Notes/07, 11-13). Runs before any heavy work: what the machine is (mounts / GCSFUSE, df, RAM, cgroup,
# CPU, a timed 64 MiB read), a phase record written at every change (e14_state.json) and a heartbeat, then the model
# copy to RAM/local disk with ranged parallel readers (verified, published atomically, fail open). The E14-P fix (raw
# reasoning in Duck's history, Notes/12) is switched on for every profile of this notebook.
import io as _e14_io
E14_DIR = Path(os.environ.get("E14_DIR", "/tmp/e14")); E14_DIR.mkdir(parents=True, exist_ok=True)
Q27X_PAYLOAD_SLUG = "esfersami50/q27-e15b-payload"


def q27x_payload(name, sha):
    """One file of the payload dataset (Notes/21: Kaggle commit runs of a 745 kB notebook never started, so the big
    payloads live in a dataset), found by its exact name under Q27X_PAYLOAD_DIR or /kaggle/input (at most 3 directory
    levels deep) and verified by sha256 before use."""
    roots = [Path(x) for x in os.environ.get("Q27X_PAYLOAD_DIR", "").split(os.pathsep) if x] + [Path("/kaggle/input")]
    seen = []
    for r in roots:
        for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name):
            for f in (sorted(r.glob(pat)) if r.is_dir() else []):
                b = f.read_bytes()
                if hashlib.sha256(b).hexdigest() == sha:
                    return b
                seen.append(str(f))
    raise FileNotFoundError(f"payload {name} (sha256 {sha[:12]}) not found or wrong under {[str(r) for r in roots]}; "
                            f"seen {seen}; attach the dataset {Q27X_PAYLOAD_SLUG}")


_E14_MOD_SHA = {"e14_loadpath.py": "7c66c179e5bb6bff7c909d0a96d4b7058c4bc0b2ff368b04367ee9f54e8fc5fc", "e14_measure.py": "ae61c9f9a03419d73dae2ba2d2aa4d5d1f4403780911832148d2dd8e8a3440d0", "e14_sglang_patches.py": "beb850abe2c072c34df0ae733642557cc44e2d808775912f291f072d29e12f11", "e14_nccl_bench.py": "355adda0d469c5ecac5abb94ff7608e2f256acc6a5dcf745e28b4d3676f03cb2", "e14_trace_summary.py": "8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46"}
with zipfile.ZipFile(_e14_io.BytesIO(q27x_payload("q27x_e14_modules_55c1eb7911c6.bin", "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040"))) as _z:
    for _n, _h in _E14_MOD_SHA.items():
        (E14_DIR / _n).write_bytes(_z.read(_n))
        assert hashlib.sha256((E14_DIR / _n).read_bytes()).hexdigest() == _h, _n
sys.path.insert(0, str(E14_DIR))
import e14_loadpath as E14L, e14_measure as E14M, e14_sglang_patches as E14P, e14_trace_summary as E14T
E14_OUT = OUT / "e14"; E14_OUT.mkdir(parents=True, exist_ok=True)
E14_RESULTS = {"build": {"variant": "VMS", "modules": _E14_MOD_SHA,
                         "payload": {"dataset": Q27X_PAYLOAD_SLUG, "q27x_e14_modules_55c1eb7911c6.bin": "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040"}},
               "phases": {}, "decisions": {}}
E14_DO_MEASURE = False   # E15P rev 2 (F10): no E14 measurement cells in this notebook
E14_STATE = E14L.E14State(WORKING / "e14_state.json")
E14_STATE.enter("PREPARE", deadline=HARD_STOP, profile=PROFILE, rerun=RERUN, measure=E14_DO_MEASURE)


def e14_save():
    try:
        write_json_atomic(E14_OUT / "e14_results.json", E14_RESULTS)
        write_json_atomic(WORKING / "e14_results.json", E14_RESULTS)
    except Exception as _e:
        print("E14 results write failed:", repr(_e)[:200], flush=True)


def _e14_heartbeat(every=60):
    while True:
        try:
            with open(WORKING / "e14_heartbeat.log", "a", encoding="utf-8") as _f:
                _f.write(f"{time.strftime('%H:%M:%S')} +{(time.perf_counter() - NOTEBOOK_T0) / 60:.1f} min "
                         f"phase {E14_STATE.rec.get('current')}\n")
        except Exception:
            pass
        time.sleep(every)


threading.Thread(target=_e14_heartbeat, daemon=True).start()
E14_RESULTS["env"] = E14L.env_report([str(SGL_MODEL_PATH), str(SGL_DRAFT_PATH), str(SGL_WHEELHOUSE), str(MODEL_PATH),
                                      "/kaggle/input"])
try:
    E14_RESULTS["read_probe"] = E14L.read_probe(SGL_MODEL_PATH) if ENGINE_BACKEND == "sglang" else None
except Exception as _e:
    E14_RESULTS["read_probe"] = {"error": repr(_e)[:200]}
print("E14 env:", json.dumps(E14_RESULTS["env"])[:1500], "| probe:", json.dumps(E14_RESULTS["read_probe"]), flush=True)
E14_STAGING = None
if ENGINE_BACKEND == "sglang" and E14L.loadpath_enabled():
    _e14_dest = tuple(x for x in os.environ.get("E14_DEST", "/dev/shm,/tmp").split(",") if x)
    E14_STAGING = E14L.Staging([SGL_MODEL_PATH, SGL_DRAFT_PATH], workers=24, stall_sec=600.0,
                               deadline=max(NOTEBOOK_T0 + 10 * 60, min(NOTEBOOK_T0 + 45 * 60, SGL_GIVE_UP_AT - 20 * 60)), log=print, state=E14_STATE,
                               dest_candidates=_e14_dest)   # E15B #27
    E14_STAGING.start()
    print("E14 staging started ->", _e14_dest, flush=True)
E14_VENV = {}


def e14_after_install(vpy, env):
    """After the SGLang venv exists: the NCCL bench (commit measurement only; the GPUs are idle while the copy runs),
    then the staged model paths (fail open: the original /kaggle/input paths)."""
    global SGL_MODEL_PATH, SGL_DRAFT_PATH
    E14_VENV.update(vpy=str(vpy), env=dict(env))
    if E14_DO_MEASURE and not E11_HARNESS_E14:
        E14_STATE.enter("NCCL_BENCH")
        _t = time.perf_counter()
        _o = E14_OUT / "nccl_bench.json"
        try:
            _r = subprocess.run([str(vpy), str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o),
                                 "--arms", "baseline,p2p_sys,shm_memcpy", "--timeout-per-arm", "240"],
                                env=env, capture_output=True, text=True, timeout=900)
            E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"rc": _r.returncode,
                                                                                  "stderr": (_r.stderr or "")[-2000:]}
        except Exception as _e:
            E14_RESULTS["nccl"] = {"error": repr(_e)[:300]}
        E14_RESULTS["nccl_sec"] = round(time.perf_counter() - _t, 1)
        e14_save()
        print("E14 NCCL bench:", json.dumps((E14_RESULTS.get("nccl") or {}).get("summary"))[:800], flush=True)
    elif E14_DO_MEASURE and E11_HARNESS_E14:
        _o = E14_OUT / "nccl_bench.json"
        subprocess.run([sys.executable, str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o), "--fake"], timeout=120)
        E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"error": "fake bench wrote nothing"}
    if E14_STAGING is not None:
        E14_STATE.enter("STAGING_JOIN")
        E14_STAGING.join(timeout=max(0.0, min(SGL_GIVE_UP_AT - 20 * 60, NOTEBOOK_T0 + 55 * 60) - time.perf_counter()))
        res = E14_STAGING.result
        if res is None:
            E14_RESULTS["staging"] = {"still_running": True, "note": "launch from the original paths"}
        else:
            paths, rep = res
            E14_RESULTS["staging"] = rep
            SGL_MODEL_PATH = Path(paths.get(str(SGL_MODEL_PATH), str(SGL_MODEL_PATH)))
            SGL_DRAFT_PATH = Path(paths.get(str(SGL_DRAFT_PATH), str(SGL_DRAFT_PATH)))
        E14_RESULTS["launch_paths"] = {"model": str(SGL_MODEL_PATH), "draft": str(SGL_DRAFT_PATH)}
        e14_save()
        print("E14 staging:", json.dumps(E14_RESULTS["staging"])[:1200], flush=True)
    E14_STATE.enter("LAUNCH", deadline=SGL_GIVE_UP_AT)


E11_HARNESS_E14 = os.environ.get("E11_HARNESS") == "1"
E14_TS = float(os.environ.get("E14_TIME_SCALE", "1"))    # the CPU harness shortens every window and phase cap; 1 on Kaggle
_ts = lambda s: max(1.5, float(s) * E14_TS)
CONFIG["e14"] = {"measure": E14_DO_MEASURE, "staging": E14_STAGING is not None,
                 "submit_arm": None if E14_DO_MEASURE else "DFlash width 6, 16 streams: mrr 17, graphs 1..17, mamba 68, min-free-slots-delay 1 (Exp 15 Notes/02)",
                 "modules": _E14_MOD_SHA}
e14_save()


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1" and E14_STAGING is None:   # E14: ranged staging replaces the cat
    try:
        import shlex as _shlex
        _wdirs = (MODEL_PATH,)   # VMS rev 2b: the FP8 reference loads first (the SGLang default read INT4 + DFlash2)
        _wfiles = sorted(str(p) for _d in _wdirs for p in Path(_d).glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    _tok_pkgs = [f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                 if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]]
    if ENGINE_BACKEND == "sglang" and before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        # E11-05: the kernel only tokenizes (S49 stack); SGLang runs in its own venv; vLLM is installed only for the fallback
        try:
            offline_pip_install(_tok_pkgs)
        except Exception as _e:
            print("tokenizer-only install failed -> the full S49 install:", repr(_e)[:300], flush=True)
            offline_pip_install(["vllm==0.19.0"])
    elif before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install(_tok_pkgs)
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if _p == "vllm" and ENGINE_BACKEND == "sglang":
            continue                                   # E11-05: checked by the fallback cell, only if it runs
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")

# E11-05: the SGLang server receives token ids made by this (S49) tokenizer. Check that the INT4 checkpoint has the same
# vocabulary (with added tokens) and tokenizes a rendered prompt identically; otherwise run the S49 vLLM engine.
E11_TOKENIZER_CHECK = {"ok": None, "skipped": ENGINE_BACKEND != "sglang"}
if ENGINE_BACKEND == "sglang":
    try:
        from tokenizers import Tokenizer as _HFTokenizer
        _t4 = _HFTokenizer.from_file(str(SGL_MODEL_PATH / "tokenizer.json"))
        _probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Example 1\nInput:\n0 1 2\n3 4 5\nOutput:\n5 4 3\n```python\ndef transform(grid):\n    return grid\n```"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort=REASONING_EFFORT)
        _probe += "Let me think.\n</think>\n\n```python\nimport numpy as np\n```<|im_end|>"
        # rev 2: + a real ARC prompt fragment (every train pair of the first test task, rendered as digit rows)
        _t0 = {"train": [{"input": [[0, 1, 2], [3, 4, 5]], "output": [[5, 4, 3], [2, 1, 0]]}]}   # E15B #17: fixed
        _probe += "".join(f"\nExample {_i + 1}\nInput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["input"])
                          + "\nOutput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["output"])
                          for _i, _p in enumerate(_t0["train"]))
        _ids_s49 = tokenizer.encode(_probe, add_special_tokens=False)
        _ids_int4 = _t4.encode(_probe, add_special_tokens=False).ids
        _same_vocab = _t4.get_vocab(with_added_tokens=True) == tokenizer.backend_tokenizer.get_vocab(with_added_tokens=True)
        # rev 2: the BPE merges too (the two files store them in different formats: "a b" strings vs [a, b] pairs)
        _norm = lambda m: [tuple(x.split(" ", 1)) if isinstance(x, str) else tuple(x) for x in m]
        _same_merges = (_norm(_load_json(SGL_MODEL_PATH / "tokenizer.json")["model"]["merges"])
                        == _norm(_load_json(MODEL_PATH / "tokenizer.json")["model"]["merges"]))
        _int4_gen = _load_json(SGL_MODEL_PATH / "generation_config.json") if (SGL_MODEL_PATH / "generation_config.json").exists() else {}
        _e4 = _int4_gen.get("eos_token_id", [])
        _same_eos = set(_e4 if isinstance(_e4, list) else [_e4]) == EOS_IDS
        E11_TOKENIZER_CHECK = {"ok": bool(_same_vocab and _same_merges and _ids_s49 == _ids_int4 and _same_eos),
                               "same_vocab": _same_vocab, "same_merges": _same_merges, "same_probe_ids": _ids_s49 == _ids_int4,
                               "probe_tokens": len(_ids_s49), "same_eos": _same_eos}
    except Exception as _e:
        E11_TOKENIZER_CHECK = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:300]}
    if not E11_TOKENIZER_CHECK["ok"]:
        print("E11 tokenizer check FAILED -> S49 vLLM FP8 engine:", E11_TOKENIZER_CHECK, flush=True)
        ENGINE_BACKEND = "vllm"
CONFIG["e11"]["tokenizer_check"] = E11_TOKENIZER_CHECK
print("E11 tokenizer check:", E11_TOKENIZER_CHECK)


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
import re, signal as _signal, urllib.request as _urlreq      # VMS: the helpers the dropped SGLang engine cell provided
def _e11_gpu_free(max_sec=180):
    """Wait until every GPU shows < 1 GiB used (nvidia-smi); returns the last reading."""
    t0, used = time.perf_counter(), None
    while time.perf_counter() - t0 < max_sec:
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=30).stdout
            used = [int(x) for x in out.split() if x.strip().isdigit()]
            if used and max(used) < 1024:
                return {"free": True, "used_mb": used, "sec": round(time.perf_counter() - t0, 1)}
        except Exception:
            return {"free": None, "error": "nvidia-smi unavailable"}
        time.sleep(3)
    return {"free": False, "used_mb": used}
# ==================================================================== CELL V1 — vLLM 0.32.0 venv, launcher, measurement
import shlex, socket, statistics, threading
VMS = {"install": None, "arms": {}, "launches": [], "order": []}
E14_RESULTS["vms"] = VMS
VMS_FAKE = os.environ.get("VMS_FAKE_SERVER")          # local CPU harness only: a fake vLLM OpenAI server script
VMS_PORT = int(os.environ.get("VMS_PORT", 8011))
VMS_URL = f"http://127.0.0.1:{VMS_PORT}"
VMS_SERVER = None
VMS_DIR = OUT / "vms"
VMS_DIR.mkdir(parents=True, exist_ok=True)


def vms_install():
    """The dataset's installer (restores the '+' Kaggle drops, venv, --no-deps), then a -lcuda shim for FlashInfer JIT."""
    if VMS_FAKE:
        return {"fake": True, "launch": [sys.executable, VMS_FAKE], "env": dict(os.environ)}
    inst = next(iter(sorted(VLLM_WHEELHOUSE.rglob("install_wheelhouse.py"))), None)
    if inst is None:
        raise FileNotFoundError(f"install_wheelhouse.py not under {VLLM_WHEELHOUSE}: attach the vLLM 0.32.0 wheelhouse")
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, str(inst), "/tmp/vllm-venv"], capture_output=True, text=True, timeout=2400)
    if r.returncode != 0:
        raise RuntimeError("vLLM wheelhouse install failed: " + (r.stdout + r.stderr)[-3000:])
    kv = dict(l.split("=", 1) for l in r.stdout.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
    shim = Path("/tmp/vms_libcuda")
    shim.mkdir(exist_ok=True)
    for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/compat", "/usr/lib64"):
        c = sorted(Path(d).glob("libcuda.so.1")) if Path(d).is_dir() else []
        if c and not (shim / "libcuda.so").exists():
            (shim / "libcuda.so").symlink_to(c[0])
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH")}
    env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"/tmp/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
               LIBRARY_PATH=f"{shim}:{env.get('LIBRARY_PATH', '')}", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               VLLM_NO_USAGE_STATS="1", VLLM_LOGGING_LEVEL="INFO", VLLM_CACHE_ROOT="/tmp/vllm-cache",
               VLLM_ALLOW_LONG_MAX_MODEL_LEN="1")
    return {"sec": round(time.perf_counter() - t0, 1), "venv_python": kv["VENV_PYTHON"],
            "launch": ["/tmp/vllm-venv/bin/vllm", "serve"], "env": env}


def vms_flags(model, *, tp=4, dp=1, seqs=16, spec=3, kv="fp8_e4m3", backend="FLASHINFER", prefix=True, mem=0.90,
              batched=4096, max_len=106496, extra=None, car_off=True):
    f = [str(model), "--tensor-parallel-size", str(tp), "--kv-cache-dtype", kv, "--max-model-len", str(max_len),
         "--max-num-seqs", str(seqs), "--max-num-batched-tokens", str(batched), "--gpu-memory-utilization", str(mem)]
    f += ["--disable-custom-all-reduce"] if car_off else []
    f += ["--seed", str(SEED), "--host", "127.0.0.1", "--port", str(VMS_PORT),
          "--served-model-name", "q27", "--attention-backend", backend, "--trust-remote-code"]
    if dp > 1:
        f += ["--data-parallel-size", str(dp)]
    f += ["--enable-prefix-caching"] if prefix else ["--no-enable-prefix-caching"]
    if spec:
        f += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": int(spec)})]
    for k, v in (extra or {}).items():
        f += [k] + ([] if v is None else [str(v)])
    return f


class VServer:
    def __init__(self, name, flags, env):
        self.name, self.log_path = name, VMS_DIR / f"vllm_server_{name}.log"
        self.cmd, self.env, self.proc = list(VMS_INST["launch"]) + list(flags), dict(env), None

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     start_new_session=True, cwd=None if VMS_FAKE else "/tmp")

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout):
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(VMS_URL + "/health", timeout=10).read()
                return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                time.sleep(3)
        return False, "timeout"

    def kill(self):
        if self.proc is None:
            return
        for sig, wait in ((_signal.SIGTERM, 30), (getattr(_signal, "SIGKILL", _signal.SIGTERM), 60)):   # rev 2b: TERM first
            try:
                if hasattr(os, "killpg"):
                    os.killpg(self.proc.pid, sig)
                else:
                    self.proc.kill()
            except Exception:
                pass
            try:
                self.proc.wait(timeout=wait)
                return
            except Exception:
                pass


def vms_prom():
    txt = _urlreq.urlopen(VMS_URL + "/metrics", timeout=15).read().decode("utf-8", "replace")
    out = {}
    for line in txt.splitlines():
        if not line or line.startswith("#"):
            continue
        m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)", line)
        if m:
            try:
                v = float(m.group(3))
            except ValueError:
                continue
            out[m.group(1)] = out.get(m.group(1), 0.0) + v
            out[m.group(1) + "@max"] = max(out.get(m.group(1) + "@max", v), v)     # rev 2b: per-engine max (DP)
    return out


def _met(p, *names):
    return sum(p.get(n, 0.0) for n in names)


class VGen(threading.Thread):
    """One streaming /v1/completions request with prompt token ids; stop() closes the stream (vLLM aborts it).
    rev 2e (GPT audits E1, HIGH-2): integer token ids from return_token_ids (completion/protocol.py:171-180,
    serving.py:417-421; logprobs.tokens are decoded strings), the terminal [DONE] and the finish_reason recorded; a request
    we did not stop that lacks either is an error, never a valid row."""
    def __init__(self, ids, max_new, temperature=1.0, ignore_eos=True, seed=None, logprobs=0):
        super().__init__(daemon=True)
        self.body = {"model": "q27", "prompt": list(ids), "max_tokens": int(max_new), "temperature": temperature,
                     "top_p": 0.95 if temperature else 1.0, "stream": True,
                     "ignore_eos": ignore_eos, "return_token_ids": True}
        if temperature:
            self.body["top_k"] = 20          # rev 2g-b: greedy sends no top_k (temperature 0 = argmax)
        if seed is not None:
            self.body["seed"] = seed
        if logprobs:
            self.body["logprobs"] = int(logprobs)      # rev 2b: the gate compares tokens and top-1 / top-2 margins
        self.text, self.err, self._resp, self._halt = [], None, None, False   # NOT _stop: Thread._stop() is a method join() calls (Py3.12)
        self.tokens, self.margins = [], []
        self.done_seen, self.finish_reason, self.first_token_at = False, None, None

    def run(self):
        try:
            req = _urlreq.Request(VMS_URL + "/v1/completions", data=json.dumps(self.body).encode(), method="POST",
                                  headers={"Content-Type": "application/json"})
            self._resp = _urlreq.urlopen(req, timeout=3600)
            for raw in self._resp:
                if self._halt:
                    break
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                payload = line[5:].strip()
                if payload == b"[DONE]":
                    self.done_seen = True
                    break
                d = json.loads(payload)
                if "error" in d or d.get("object") == "error":     # rev 2b: an in-stream error event (serving.py:497-503)
                    self.err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:200]
                    break
                for c in d.get("choices") or []:
                    self.text.append(c.get("text") or "")
                    ids_ = c.get("token_ids")
                    lp = c.get("logprobs") or {}
                    if ids_ is None:
                        if c.get("text") or lp.get("tokens"):
                            self.err = "MISSING_STREAM_TOKEN_IDS"
                    elif any(not isinstance(v, int) or isinstance(v, bool) for v in ids_):
                        self.err = "INVALID_STREAM_TOKEN_IDS"
                    else:
                        if ids_ and self.first_token_at is None:
                            self.first_token_at = time.perf_counter()
                        self.tokens.extend(ids_)
                    for top in lp.get("top_logprobs") or []:
                        vals = sorted((v for v in (top or {}).values() if isinstance(v, (int, float))), reverse=True)
                        self.margins.append(round(vals[0] - vals[1], 4) if len(vals) >= 2 else None)
                    if c.get("finish_reason") is not None:
                        self.finish_reason = c["finish_reason"]
                if self.err:
                    break
        except Exception as e:
            if not self._halt:
                self.err = repr(e)[:200]
        if not self._halt and not self.err and not (self.done_seen and self.finish_reason):
            self.err = "INCOMPLETE_SSE: [DONE] seen %s, finish_reason %s" % (self.done_seen, self.finish_reason)

    def stop(self):
        self._halt = True
        try:      # rev 2b: shut the socket down first, so a reader blocked in recv returns (close() alone can wait on it)
            self._resp.fp.raw._sock.shutdown(socket.SHUT_RDWR)
        except Exception:
            pass
        try:
            if self._resp is not None:
                self._resp.close()
        except Exception:
            pass

    def row(self):
        return {"text": "".join(self.text), "tokens": list(self.tokens), "margins": list(self.margins), "error": self.err,
                "done_seen": self.done_seen, "finish_reason": self.finish_reason}


def _vms_dead():
    return VMS_SERVER is not None and not VMS_SERVER.alive()


def vms_idle(wait=60.0):
    """rev 2e (GPT audit E6): 0 running and 0 waiting in 3 consecutive samples (2 s apart) within `wait`."""
    end, hits = time.perf_counter() + wait, 0
    while time.perf_counter() < end:
        try:
            p = vms_prom()
            hits = hits + 1 if (_met(p, "vllm:num_requests_running") == 0 and _met(p, "vllm:num_requests_waiting") == 0) else 0
        except Exception:
            hits = 0
        if hits >= 3:
            return True
        time.sleep(2)
    return False


def vms_window(prompts, level, secs, tag, deadline=None):
    """`level` streams; wait until vllm:num_requests_running reaches level, a stable level - 1, or a KV plateau (requests
    waiting while running stays flat for 30 s); then wait for the prefill to settle; then measure the counters.
    rev 2b: every wait ends at `deadline` (the arm's own: the 2 x TP2 reserve stays intact) and on a dead server.
    rev 2e (GPT audits E5-E7, HIGH-5, E9): the prefill must settle (3 equal prompt-token readings) or the window is
    PREFILL_NOT_SETTLED; the streams are joined and the server must be idle afterwards (else DIRTY_SERVER_AFTER_WINDOW);
    every started stream is checked for empty / degenerate text; steady decode tok/s and end-to-end output tok/s (from
    submission) are both kept; prefix-cache hits / queries / cached prompt tokens are recorded."""
    end = min(HARD_STOP - 120, deadline or HARD_STOP)
    if end - time.perf_counter() < secs + 60:
        return {"skipped": "NOT_TESTED_BUDGET: deadline"}
    # rev 2g: max_tokens up to 32k inside max-model-len 106,496 (a stream that finished inside the window lowered
    # running_mean: a false reject)
    gens = [VGen(prompts[i % len(prompts)], max(1024, min(32768, 106496 - len(prompts[i % len(prompts)]) - 512)))
            for i in range(level)]
    out = {"tag": tag, "level": level}
    p_sub = p0 = p1 = None
    t_sub = t0 = t1 = None
    try:
        try:
            p_sub = vms_prom()
        except Exception:
            p_sub = {}
        t_sub = time.perf_counter()
        for g in gens:
            g.start()
        t_w, got, best, hist, plateau = time.perf_counter(), None, 0, [], False
        while time.perf_counter() < min(t_w + 900, end - secs - 30):
            if _vms_dead():
                out["error"] = "the server died while the streams were admitted"
                return out
            try:
                p = vms_prom()
                run, wait = _met(p, "vllm:num_requests_running"), _met(p, "vllm:num_requests_waiting")
            except Exception:
                run, wait = 0, 0
            best = max(best, run)
            hist.append((time.perf_counter(), run))
            if run >= level:
                got = level
                break
            alive = sum(1 for g in gens if g.is_alive())
            if alive < level - 1 or (alive < level and run >= alive):      # rev 2e-e / 2g: level (or level - 1) unreachable
                out["streams_ended_before_level"] = level - alive
                break
            flat = [r for t_, r in hist if t_ >= hist[-1][0] - 30]
            if wait > 0 and run >= 1 and hist[-1][0] - hist[0][0] >= 30 and len(set(flat)) == 1:
                got, plateau = int(run), True        # KV-bound: measure what the server really holds
                break
            time.sleep(2)
        if got is None and best >= level - 1:
            got = int(best)
        out.update(measured_level=got, plateau=plateau, capacity_limited=plateau, wait_sec=round(time.perf_counter() - t_w, 1))
        if got is None:
            out["error"] = f"the server never ran {level} requests (max {best})"
            return out
        # a request counts as running once its first prefill chunk is scheduled: start only when nothing waits (or the
        # plateau) and 3 consecutive prompt-token readings are equal. Up to 300 s: a no-prefix-cache arm prefills
        # 16 x 40k = 640k tokens (minutes); a window that still did not settle is measured but labelled mixed_load
        # (excluded from measurement_ok), never thrown away
        t_s, last, stable, settled = time.perf_counter(), None, 0, False
        while time.perf_counter() - t_s < 300 and time.perf_counter() < end - secs - 15:
            try:
                p = vms_prom()
                pt, wait = _met(p, "vllm:prompt_tokens_total"), _met(p, "vllm:num_requests_waiting")
            except Exception:
                break
            stable = stable + 1 if (last is not None and pt == last and (wait == 0 or plateau)) else 0
            if stable >= 2:
                settled = True
                break
            last = pt
            time.sleep(2)
        out.update(settle_sec=round(time.perf_counter() - t_s, 1), prefill_settled=settled, mixed_load=not settled)
        p0, t0 = vms_prom(), time.perf_counter()
        runs, kvs, runs_mx = [], [], []
        while time.perf_counter() - t0 < secs:
            time.sleep(2)
            if _vms_dead():
                out["error"] = "the server died inside the window"
                return out
            try:
                p = vms_prom()
                runs.append(_met(p, "vllm:num_requests_running"))
                runs_mx.append(p.get("vllm:num_requests_running@max", 0.0))      # rev 2g-b: per engine (DP skew)
                kvs.append(p.get("vllm:kv_cache_usage_perc@max", 0.0))      # rev 2b: per engine, never the DP sum
            except Exception:
                pass
        p1, t1 = vms_prom(), time.perf_counter()
        out["_runs"], out["_kvs"], out["_runs_mx"] = runs, kvs, runs_mx
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"[:300]
        return out
    finally:
        for g in gens:
            g.stop()
        for g in gens:
            if g.ident is not None:
                g.join(timeout=5)
        out["threads_not_joined"] = sum(1 for g in gens if g.is_alive())
        out["idle_after"] = (not _vms_dead()) and vms_idle(60)
        if out["threads_not_joined"] or not out["idle_after"]:
            out["error"] = out.get("error") or "DIRTY_SERVER_AFTER_WINDOW"
        started = [g for g in gens if g.first_token_at is not None]
        texts = ["".join(g.text) for g in started]
        out.update(finished_empty=sum(1 for g in gens if g.done_seen and not g.tokens),      # rev 2g-c (agent 3 B2)
                   no_first_token=len(gens) - len(started), empty_outputs=sum(1 for x in texts if not x.strip()),
                   degenerate=vms_degenerate(texts), stream_errors=sum(1 for g in gens if g.err),
                   stream_error_samples=[g.err for g in gens if g.err][:3])
    runs, kvs = out.pop("_runs", []), out.pop("_kvs", [])
    out["_runs_mx"] = out.get("_runs_mx") or []
    dt = t1 - t0
    gen = _met(p1, "vllm:generation_tokens_total") - _met(p0, "vllm:generation_tokens_total")
    acc = _met(p1, "vllm:spec_decode_num_accepted_tokens_total") - _met(p0, "vllm:spec_decode_num_accepted_tokens_total")
    drf = _met(p1, "vllm:spec_decode_num_drafts_total") - _met(p0, "vllm:spec_decode_num_drafts_total")
    gen_all = _met(p1, "vllm:generation_tokens_total") - _met(p_sub, "vllm:generation_tokens_total")
    out.update(window_sec=round(dt, 1), tok_s=round(gen / dt, 1) if dt > 0 else None,
               end_to_end_tok_s=round(gen_all / (t1 - t_sub), 1) if t1 and t_sub and t1 > t_sub else None,
               accept=round(1 + acc / drf, 2) if drf > 0 else None, running_mean=round(statistics.mean(runs), 2) if runs else None,
               running_engine_max_mean=round(statistics.mean(out.get("_runs_mx") or [0]), 2),
               kv_max=round(max(kvs), 3) if kvs else None,
               preemptions=_met(p1, "vllm:num_preemptions_total") - _met(p0, "vllm:num_preemptions_total"),
               prefix_cache_hits=_met(p1, "vllm:prefix_cache_hits_total") - _met(p_sub, "vllm:prefix_cache_hits_total"),
               prefix_cache_queries=_met(p1, "vllm:prefix_cache_queries_total") - _met(p_sub, "vllm:prefix_cache_queries_total"),
               prompt_tokens_cached=_met(p1, "vllm:prompt_tokens_cached_total") - _met(p_sub, "vllm:prompt_tokens_cached_total"))
    return out


def vms_window_healthy(w, level):
    """rev 2e (2nd audit HIGH-3): speed is evidence only from a clean window that held its level."""
    return bool(w) and not w.get("error") and not w.get("skipped") and (w.get("tok_s") or 0) > 0 \
        and w.get("prefill_settled") is True and w.get("stream_errors", 0) == 0 and w.get("empty_outputs", 0) == 0 and w.get("degenerate", 0) == 0 \
        and (w.get("measured_level") or 0) >= level and (w.get("running_mean") or 0) >= 0.80 * level \
        and w.get("no_first_token", level) == 0 and w.get("threads_not_joined", 1) == 0 and w.get("idle_after") is True \
        and (w.get("preemptions") or 0) == 0 and w.get("finished_empty", 0) == 0


def vms_greedy(prompts, n_tok=256, concurrent=False, deadline=None):
    """Greedy rows {text, token ids, top-1 / top-2 margins, error, [DONE], finish_reason}; rev 2b: one deadline for the
    whole pass; a row still running at the deadline is stopped and marked."""
    end = min(HARD_STOP - 120, deadline or HARD_STOP)
    gens = [VGen(p, n_tok, temperature=0.0, ignore_eos=False, logprobs=2) for p in prompts]
    if concurrent:
        for g in gens:
            g.start()
        for g in gens:
            g.join(max(0.0, min(900, end - time.perf_counter())))
    else:
        for g in gens:
            if end - time.perf_counter() < 30:
                g.err = "skipped: arm deadline"
                continue
            g.start()
            g.join(max(0.0, min(900, end - time.perf_counter())))
    for g in gens:
        if g.is_alive():
            g.stop()
            g.err = g.err or "stopped: arm deadline"
    return [g.row() for g in gens]


def vms_degenerate(texts):
    """rev 2 (GPT audit V1, vLLM #53912: silent empty / repeated-punctuation output with a normal finish): empty text, a
    run of 32+ identical characters, or >= 200 characters with < 8 distinct characters or < 10 % letters / digits.
    (A distinct-character RATIO would flag every long normal text: 8k characters of prose use ~100 distinct ones.)"""
    return sum(1 for x in texts if vms_text_bad(x))


def vms_grid_like(s):
    """rev 2g (GPT R2F-06): an ARC grid by structure ("0 0 1\n0 0 0", "[[0, 1], ...]"): only digits / brackets / commas /
    whitespace and every row <= 30 cells (the ARC maximum). "0" * 250 or "0 0 0 ..." x 200 is not a grid."""
    s = (s or "").strip()
    return bool(re.fullmatch(r"[0-9,\[\]\s]+", s)) and len(re.findall(r"\d", s)) >= 2 \
        and all(len(re.findall(r"\d", r_)) <= 30 for r_ in re.split(r"\]|\n", s))


def vms_text_bad(x):
    """rev 2e-c: one text is degenerate when empty, or (not grid-like) a 32-character run of one character, < 8 distinct
    characters over 200, or < 10 % letters / digits. A grid is never degenerate (it repeats by nature)."""
    s = (x or "").strip()
    if not s:
        return True
    if vms_grid_like(s):
        return False
    if re.search(r"(.)\1{31,}", s):
        return True
    return len(s) >= 200 and (len(set(s)) < 8 or sum(ch.isalnum() for ch in s) / len(s) < 0.10)


def vms_kernels(txt):
    """rev 2 (GPT audit V6): the GEMM / attention kernels vLLM says it selected (never assume the default)."""
    seen = []
    for ln in txt.splitlines():
        m = re.search(r"((?:Selected|Using|Chose)\b.{0,60}?(?:kernel|backend).{0,80}|--linear-backend=.{0,160})", ln, re.I)
        if m and m.group(1).strip() not in seen:
            seen.append(m.group(1).strip())
    return seen[:30]


def vms_tail_bad(ids):
    """rev 2e (GPT audits E2 / HIGH-10), rev 2e-c: the continuation after the first divergence, decoded, is degenerate by
    the shared text rule (a grid is not; empty, repeated punctuation, near-alphabet-free text is)."""
    ids = list(ids or [])
    if len(ids) < 16:
        return False          # too short to judge
    try:
        return vms_text_bad(tokenizer.decode(ids, skip_special_tokens=True))
    except Exception:
        return False


def vms_diverge(a, b):
    """rev 2b (agent 3 F10): rows compared by integer token id (rev 2e) with the EMS compare_passes rule. A large-margin
    flip = a divergence where either run was confident (top-1 minus top-2 > 1 nat); a divergence without both margins is
    missing evidence, never "no flip"; an errored, incomplete (no [DONE] / finish_reason) or empty row is an error."""
    expected = max(len(a), len(b))
    per = []
    for i in range(expected):
        ra = a[i] if i < len(a) else {"error": "missing row"}
        rb = b[i] if i < len(b) else {"error": "missing row"}
        ia, ib = list(ra.get("tokens") or []), list(rb.get("tokens") or [])
        inc = [x for x in (ra, rb) if "done_seen" in x and not (x.get("done_seen") and x.get("finish_reason"))]
        if ra.get("error") or rb.get("error") or inc or not ia or not ib:
            per.append({"error": ra.get("error") or rb.get("error") or ("incomplete stream" if inc else "no tokens")})
            continue
        if ia == ib:
            per.append({"identical": True})
            continue
        k = next((j for j, (u, v) in enumerate(zip(ia, ib)) if u != v), min(len(ia), len(ib)))
        ma = (ra.get("margins") or [None] * (k + 1))[k] if k < len(ra.get("margins") or []) else None
        mb = (rb.get("margins") or [None] * (k + 1))[k] if k < len(rb.get("margins") or []) else None
        per.append({"identical": False, "first_divergence": k, "margin_a": ma, "margin_b": mb,
                    "missing_margin": ma is None or mb is None,
                    "large_margin_flip": ma is not None and mb is not None and max(ma, mb) > 1.0,
                    "tail_bad": vms_tail_bad(ib[k:])})
    ok = [x for x in per if "identical" in x]
    div = [x for x in ok if not x["identical"]]
    return {"expected": expected, "compared": len(ok), "errors": expected - len(ok),
            "identical": sum(1 for x in ok if x["identical"]), "differ": len(div),
            "large_margin_flips": sum(1 for x in div if x["large_margin_flip"]),
            "missing_margins": sum(1 for x in div if x["missing_margin"]),
            "tail_bad": sum(1 for x in div if x.get("tail_bad")),
            "first_divergence": [x.get("first_divergence") for x in per]}


VMS_INST = None
if VMS_ON:
    try:
        VMS_INST = vms_install()
        VMS["install"] = {k: v for k, v in VMS_INST.items() if k != "env"}
        print("VMS install:", json.dumps(VMS["install"])[:600], flush=True)
    except Exception as _e:
        VMS["install"] = {"error": f"{type(_e).__name__}: {_e}"[:2000]}
        print("VMS install FAILED:", VMS["install"]["error"][:1500], flush=True)
e14_save()


In [ ]:
# ==================================================================== CELL V2 — the vLLM arms
VMS_INT4 = SGL_MODEL_PATH          # RedHat INT4 (compressed-tensors) Kaggle Model; MODEL_PATH = the FP8 repack
VMS_RESERVE = 0.0


def vms_launch(name, flags, env_extra=None, force=False):
    global VMS_SERVER
    rec = {"arm": name, "flags": list(flags), "env": env_extra or {}}
    if HARD_STOP - time.perf_counter() - (0 if force else VMS_RESERVE) - 8 * 60 < 12 * 60:
        rec["skipped"] = "NOT_TESTED_BUDGET: reserved 2 x TP2 slot / hard stop"
        VMS["launches"].append(rec)
        return False, rec
    if VMS_SERVER is not None:
        VMS_SERVER.kill()
        VMS_SERVER = None
    if not VMS_FAKE:
        rec["gpu_free"] = _e11_gpu_free()
    left = HARD_STOP - time.perf_counter() - (0 if force else VMS_RESERVE) - 8 * 60     # rev 2b: after the kill + GPU wait
    if left < 10 * 60:
        rec["skipped"] = "NOT_TESTED_BUDGET: reserved 2 x TP2 slot / hard stop (after the GPU wait)"
        VMS["launches"].append(rec)
        return False, rec
    env = dict(VMS_INST["env"])
    env.update(env_extra or {})
    srv = VServer(name, flags, env)
    srv.start()
    try:          # rev 2g: a crash while this server starts never leaves it holding the GPUs
        ok, why = srv.wait_ready(min(30 * 60, left))
    except BaseException:
        try:
            srv.kill()
        except Exception:
            pass
        raise
    rec.update(ok=ok, why=why, sec=round(time.perf_counter() - srv.t0, 1))
    txt = srv.log_text()
    m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", txt)
    rec["kv_tokens_all"] = [int(x.replace(",", "")) for x in m]          # rev 2b: DP = one line per engine
    rec["kv_tokens"] = min(rec["kv_tokens_all"]) if m else None
    if ok and not m:      # rev 2g-b: a /health 200 without this server's own KV line = another listener on the port
        ok, why = False, "ready, but no 'GPU KV cache size' line in this server's log (another process on the port?)"
    m = re.findall(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)x", txt)
    rec["max_concurrency_at_max_len"] = float(m[-1]) if m else None
    rec["kernels"] = vms_kernels(txt)
    if ok:
        VMS_SERVER = srv
    else:
        rec["log_tail"] = txt[-4000:]
        srv.kill()
    VMS["launches"].append(rec)
    e14_save()
    print(f"VMS launch {name}: {why} | KV {rec.get('kv_tokens')}", flush=True)
    return ok, rec


VMS_REF = {}


def vms_int4_mtp_static(model):
    """rev 2 (GPT audit V3, vLLM #58807: a W4A16 checkpoint with dense BF16 MTP crashed at load because MTP was not in
    quantization_config.ignore, so the predictor was built quantized). vLLM 0.32.0 builds the predictor under prefix
    'mtp' (qwen3_5_mtp.py:233-235) with the target's quant config (:93-105), so every dense mtp.* weight must be
    exempted by the ignore list. CPU only: config.json + the safetensors index, no weights read."""
    out = {"model": str(model)}
    try:
        cfg = json.loads((Path(model) / "config.json").read_text(encoding="utf-8"))
        q = cfg.get("quantization_config") or (cfg.get("text_config") or {}).get("quantization_config") or {}
        ign = [str(x) for x in (q.get("ignore") or [])]

        def ignored(name):
            # rev 2b: vLLM 0.32.0 matches the fused name by exact name or re.match (config_utils.py:344-362, 392-412)
            return any(re.match(x[3:], name) if x.startswith("re:") else name == x for x in ign)
        wm = json.loads((Path(model) / "model.safetensors.index.json").read_text(encoding="utf-8"))["weight_map"]
        mtp = [k for k in wm if k.startswith("mtp.")]
        packed = [k for k in mtp if any(s in k for s in ("weight_packed", "weight_scale", "weight_zero_point"))]
        mods = sorted({k.rsplit(".", 1)[0] for k in mtp if k.endswith(".weight") and ("proj" in k or k.startswith("mtp.fc"))})
        _fuse = {"q_proj": "qkv_proj", "k_proj": "qkv_proj", "v_proj": "qkv_proj", "gate_proj": "gate_up_proj", "up_proj": "gate_up_proj"}
        mods = sorted(set(mods) | {m.rsplit(".", 1)[0] + "." + _fuse[m.rsplit(".", 1)[1]] for m in mods if m.rsplit(".", 1)[1] in _fuse})
        not_ign = [m for m in mods if not ignored(m)]
        out.update(format=q.get("format"), method=q.get("quant_method"), mtp_keys=len(mtp), packed=len(packed),
                   dense_linear_modules=len(mods), not_ignored=not_ign[:10],
                   mtp_ignore_entries=[x for x in ign if "mtp" in x])
        # rev 2g (GPT R4): a packed MTP module that the ignore list exempts would be built dense (wrong shapes)
        packed_mods = sorted({k.rsplit(".", 1)[0] for k in packed})
        packed_ign = [m for m in packed_mods if ignored(m)]
        out.update(packed_modules=len(packed_mods), packed_but_ignored=packed_ign[:10])
        out["ok"] = bool(mtp) and not not_ign and not packed_ign        # rev 2b: un-ignored dense linears = #58807
        out["why"] = "MTP dense and exempted by the ignore list" if out["ok"] and not packed else (
            "MTP stored quantized" if out["ok"] else "dense MTP linears missing from the ignore list (#58807 crash)")
    except Exception as e:
        out.update(ok=False, why=f"static check failed: {type(e).__name__}: {e}"[:300])
    VMS["int4_mtp_static"] = out
    e14_save()
    print("VMS INT4 MTP static check:", json.dumps(out)[:600], flush=True)
    return out["ok"]


VMS_ROWS = {}          # rev 2: each A/A arm's own greedy rows (the production reference and the oracle)


def _texts(rows):
    return [r.get("text") or "" for r in rows]


def vms_save_rows(tag, rows):
    """rev 2b (agent 3 F10): every greedy row kept, so any verdict can be re-judged after the run."""
    try:
        with open(VMS_DIR / "vms_greedy_rows.jsonl", "a", encoding="utf-8") as fh:
            fh.write(json.dumps({"tag": tag, "rows": rows}) + "\n")
    except Exception:
        pass


def vms_aa(name, deadline=None):
    rows = {"serial_a": vms_greedy(_S[:6], deadline=deadline), "serial_b": vms_greedy(_S[:6], deadline=deadline),
            "conc_a": vms_greedy(_S, concurrent=True, deadline=deadline), "conc_b": vms_greedy(_S, concurrent=True, deadline=deadline)}
    if _X:      # rev 2e (GPT audits MEDIUM-7 / E11): a greedy parity reference at ~85k context too
        rows["xl_a"] = vms_greedy(_X[:4], concurrent=True, deadline=deadline)
        rows["xl_b"] = vms_greedy(_X[:4], concurrent=True, deadline=deadline)
    vms_save_rows(name + "_aa", rows)
    s, c = vms_diverge(rows["serial_a"], rows["serial_b"]), vms_diverge(rows["conc_a"], rows["conc_b"])
    complete = all(x["expected"] and x["compared"] == x["expected"] and not x["missing_margins"] for x in (s, c)) \
        and s["expected"] >= min(6, len(_S)) and c["expected"] >= len(_S)       # rev 2g (GPT R5): the planned rows
    return rows, {"serial": s, "conc": c, "complete": complete,
                  "xl": vms_diverge(rows["xl_a"], rows["xl_b"]) if _X else None,
                  "degenerate": vms_degenerate(_texts(rows["serial_a"] + rows["serial_b"] + rows["conc_a"] + rows["conc_b"]))}


def vms_judge(s, c, base, gate=True):
    """The EMS parity rule on integer-token rows against `base` (the oracle once it exists): large-margin flips may not
    exceed base's own A/A flips (serial and all at once); no more degenerate outputs than base's A/A; rev 2e: no more
    degenerated continuations after a divergence than base's A/A; INCOMPLETE when any compared row errored, was
    incomplete or a divergence has no margins (fail closed). gate_class: PASS_EXACT = every sequence identical,
    BOUNDARY = a PASS with only near-tie divergences (adoptable under the documented tolerance, not exact: confirm on ARC
    tasks; certified_exact = adoptable and PASS_EXACT)."""
    cs, aa_s = vms_diverge(base["serial_a"], s), vms_diverge(base["serial_a"], base["serial_b"])
    cc, aa_c = vms_diverge(base["conc_a"], c), vms_diverge(base["conc_a"], base["conc_b"])
    dg, dg_aa = vms_degenerate(_texts(list(s) + list(c))), vms_degenerate(_texts(base["serial_a"] + base["conc_a"]))
    d = {"serial": cs, "conc": cc, "aa_serial": aa_s, "aa_conc": aa_c, "degenerate": dg, "aa_degenerate": dg_aa}
    if not gate:
        return "diagnostic", d
    if any(x["errors"] or x["missing_margins"] or not x["compared"] for x in (cs, aa_s, cc, aa_c)) \
            or cs["expected"] < min(6, len(_S)) or cc["expected"] < len(_S):       # rev 2g (GPT R5): the planned rows
        d["gate_class"] = "INCOMPLETE"
        return "INCOMPLETE", d
    passed = cs["large_margin_flips"] <= aa_s["large_margin_flips"] and cc["large_margin_flips"] <= aa_c["large_margin_flips"] \
        and dg <= dg_aa and cs["tail_bad"] + cc["tail_bad"] <= aa_s["tail_bad"] + aa_c["tail_bad"]
    d["gate_class"] = ("PASS_EXACT" if cs["identical"] == cs["compared"] and cc["identical"] == cc["compared"] else "BOUNDARY") \
        if passed else "FAIL"
    return ("PASS" if passed else "FAIL"), d


def vms_judge_xl(x, base):
    """rev 2e: the ~85k greedy rows against the oracle's own XL A/A (flips and tails decide, as above)."""
    if not base.get("xl_a") or not x:
        return None
    cx, aa = vms_diverge(base["xl_a"], x), vms_diverge(base["xl_a"], base["xl_b"])
    if any(r["errors"] or r["missing_margins"] or not r["compared"] for r in (cx, aa)):
        return {"verdict": "INCOMPLETE", "cmp": cx, "aa": aa}
    ok = cx["large_margin_flips"] <= aa["large_margin_flips"] and cx["tail_bad"] <= aa["tail_bad"]
    return {"verdict": "PASS" if ok else "FAIL", "cmp": cx, "aa": aa}


VMS_CANARY = {}


def vms_canary(level, deadline=None):
    """rev 2e (GPT audit MEDIUM-9, SGLang #36548 cross-request bleed): `level` concurrent greedy requests, each asked to
    repeat its own random sentinel (thinking off); a sentinel found in ANOTHER request's output is a leak."""
    import random as _rnd
    rng = _rnd.Random(SEED + level)
    sent = ["Z%04d-%s" % (i, "".join(rng.choice("QWXJVK") for _ in range(6))) for i in range(level)]
    ids = [tokenizer.encode(tokenizer.apply_chat_template([{"role": "user", "content":
           f"Repeat this code exactly once and nothing else: {s_}"}], tokenize=False, add_generation_prompt=True,
           enable_thinking=False), add_special_tokens=False) for s_ in sent]
    rows = vms_greedy(ids, n_tok=32, concurrent=True, deadline=deadline)
    own = sum(1 for s_, r in zip(sent, rows) if s_ in (r.get("text") or ""))
    leaks = sum(1 for i, r in enumerate(rows) for j, s_ in enumerate(sent) if j != i and s_ in (r.get("text") or ""))
    c_ = {"level": level, "own_hits": own, "foreign_leaks": leaks,
          "errors": sum(1 for r in rows if r.get("error")) + max(0, level - len(rows))}
    # rev 2g (GPT R2F-07): 0 own hits = the detector showed nothing: INCONCLUSIVE, not proof of isolation
    c_["informative"] = own > 0 and c_["errors"] == 0
    c_["classification"] = ("FAIL" if leaks else "INCOMPLETE" if c_["errors"] else
                            "INCONCLUSIVE" if not c_["informative"] else "PASS_NO_OBSERVED_LEAK")
    return c_


def vms_classify(a):
    """rev 2e (GPT audits E8 / HIGH-3): speed, correctness, capacity and adoption kept apart. Re-run for every arm at the
    end (the production reference is judged only once the oracle exists)."""
    level = a.get("level_wanted") or 16
    a["measurement_ok"] = all(vms_window_healthy(a.get(b), level) for b in ("short", "long"))
    a["capacity_ok"] = all(((a.get(b) or {}).get("measured_level") or 0) >= level and not (a.get(b) or {}).get("capacity_limited")
                           for b in ("short", "long"))
    # rev 2g (GPT R1 / R2F-01): missing evidence never counts as a pass: XL PASS whenever XL prompts exist; the canary
    # present, at this level, 0 errors / 0 leaks; DP rows diagnostic; a PCIe-IPC arm must show the init line
    _c = a.get("canary")
    _canary_ok = isinstance(_c, dict) and _c.get("level") == level and _c.get("errors") == 0 and _c.get("foreign_leaks") == 0
    _xl_ok = a.get("xl_gate") == "PASS" if _X else a.get("xl_gate") in (None, "PASS")
    a["backend_unverified"] = bool((a.get("env") or {}).get("VLLM_ALLREDUCE_USE_FLASHINFER_PCIE_IPC")
                                   and not (a.get("pcie_ipc") or {}).get("initialized"))
    a["correctness_ok"] = a.get("gate") == "PASS" and _xl_ok and _canary_ok
    a["adoptable"] = bool(VMS.get("oracle_ready") and a["measurement_ok"] and a["capacity_ok"] and a["correctness_ok"]
                          and not a.get("diagnostic_only") and not a["backend_unverified"]
                          and a.get("server_alive_after") and not a.get("log_errors") and not a.get("error"))
    a["candidate_for_arc_validation"] = a["adoptable"]
    a["production_approved"] = False         # rev 2g: needs the 60k+ generation and ARC task gates (not in this matrix)
    a["certified_exact"] = a["adoptable"] and a.get("gate_class") == "PASS_EXACT"     # rev 2e-d
    a["ok"] = a["measurement_ok"]        # "speed data from clean windows exists"; adoption is a["adoptable"]
    return a


def _e22_trace_summary(pdir):
    """rank-0 (or the first) torch profiler trace: CUDA kernel time per category, the decode-step count, top kernels"""
    import gzip
    files = sorted(Path(pdir).rglob("*.pt.trace.json*"), key=lambda p: p.stat().st_mtime)
    if not files:
        return {"error": "no trace file under " + str(pdir), "listing": [str(p) for p in Path(pdir).rglob("*")][:20]}
    f = next((p for p in files if "rank0" in p.name or "rank-0" in p.name or "_0." in p.name), files[0])
    opener = gzip.open if f.name.endswith(".gz") else open
    with opener(f, "rt", encoding="utf-8") as fh:
        tr = json.load(fh)
    ev = [e for e in tr.get("traceEvents", []) if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("kernel", "gpu_kernel")]
    cats = {"nccl_allreduce": ("allreduce", "all_reduce"), "nccl_allgather": ("allgather", "all_gather"),
            "nccl_other": ("nccl",), "gemm_marlin": ("marlin",), "gemm_other": ("gemm", "cutlass", "sm89_xmma", "ampere", "cublas"),
            "gdn_linear_attn": ("gdn", "fla", "chunk_", "recurrent", "fused_recurrent", "gated_delta", "causal_conv"),
            "attention": ("flash", "attn", "attention", "paged"), "sampling": ("sampl", "topk", "top_k", "argmax", "softmax")}
    tot, per, names = 0.0, {k: 0.0 for k in list(cats) + ["other"]}, {}
    for e in ev:
        n, d = str(e.get("name", "")).lower(), float(e.get("dur", 0.0))
        tot += d
        names[e.get("name", "")] = names.get(e.get("name", ""), 0.0) + d
        for k, pats in cats.items():
            if any(p in n for p in pats):
                per[k] += d
                break
        else:
            per["other"] += d
    span = (max(e["ts"] + e.get("dur", 0) for e in ev) - min(e["ts"] for e in ev)) if ev else 0.0
    return {"file": f.name, "files": len(files), "kernels": len(ev), "kernel_us_total": round(tot), "span_us": round(span),
            "by_category_us": {k: round(v) for k, v in per.items()},
            "by_category_pct": {k: round(100 * v / tot, 1) for k, v in per.items()} if tot else {},
            "top_kernels_us": dict(sorted(((k, round(v)) for k, v in names.items()), key=lambda x: -x[1])[:40])}


def _safe_profile(name, flags, level):
    """rev 2b: one profiled launch; any failure is recorded on the arm, the server killed, the plan goes on"""
    global VMS_SERVER
    a = {"ok": False, "level_wanted": level}
    VMS["arms"][name] = a
    VMS["order"].append(name)
    try:
        pdir = VMS_DIR / "profile"
        pdir.mkdir(parents=True, exist_ok=True)
        fl = list(flags) + ["--profiler-config", json.dumps({"profiler": "torch", "torch_profiler_dir": str(pdir)})]
        ok, rec = vms_launch(name, fl)
        a.update(flags=fl, launch_sec=rec.get("sec"), skipped=rec.get("skipped"), kv_tokens=rec.get("kv_tokens"))
        if not ok:
            return a
        box = {}
        th = threading.Thread(target=lambda: box.update(w=vms_window(_L, level, _ts(150), name + "_long",
                                                                       deadline=HARD_STOP - 600)), daemon=True)
        th.start()
        t0, steady = time.perf_counter(), 0
        while th.is_alive() and time.perf_counter() - t0 < _ts(900):
            time.sleep(_ts(5))
            try:
                p = vms_prom()
            except Exception:
                continue
            run, wait = p.get("vllm:num_requests_running", 0.0), p.get("vllm:num_requests_waiting", 0.0)
            steady = steady + 1 if (run >= level - 1 and wait == 0) else 0
            if steady >= 6:          # ~30 s at the level with no prefill queue
                break
        a["steady_before_profile"] = steady
        for ep in ("/start_profile", "/stop_profile"):
            try:
                _urlreq.urlopen(_urlreq.Request(VMS_URL + ep, data=b"", method="POST"), timeout=600).read()
                a[ep.strip("/")] = "ok"
            except Exception as e:
                a[ep.strip("/")] = repr(e)[:200]
            if ep == "/start_profile":
                time.sleep(_ts(6))
        time.sleep(_ts(20))          # the trace is written by every rank after stop
        th.join(timeout=_ts(600))
        a["window"] = {k: v for k, v in (box.get("w") or {}).items() if k in ("tok_s", "accept", "measured_level", "error")}
        a["trace"] = _e22_trace_summary(pdir)
        a["ok"] = "error" not in a["trace"]
    except Exception as e:
        a["error"] = f"{type(e).__name__}: {e}"[:500]
    finally:
        try:
            if VMS_SERVER is not None:
                VMS_SERVER.kill()
            VMS_SERVER = None
        except Exception:
            pass
        e14_save()
        print("VMS arm", name, json.dumps({k: v for k, v in a.items() if k != "flags"}, default=str)[:1500], flush=True)
    return a


def vms_arm(name, flags, **kw):
    """rev 2f: one arm's own crash never ends the remaining arms (recorded, the server killed, the results saved)."""
    try:
        return _vms_arm(name, flags, **kw)
    except Exception as _e:
        a = dict(VMS["arms"].get(name) or {}, ok=False, adoptable=False, certified_exact=False,
                 error=f"arm crashed: {type(_e).__name__}: {_e}"[:500])
        VMS["arms"][name] = a
        if name not in VMS["order"]:
            VMS["order"].append(name)
        print(f"VMS arm {name} crashed:", traceback.format_exc()[-1500:], flush=True)
        try:
            if VMS_SERVER is not None:
                VMS_SERVER.kill()
        except Exception:
            pass
        e14_save()
        return a


def _vms_arm(name, flags, *, level=16, gate=True, env_extra=None, cap=14 * 60):
    ok, rec = vms_launch(name, flags, env_extra)
    a = {"ok": False, "launch_sec": rec.get("sec"), "kv_tokens": rec.get("kv_tokens"), "skipped": rec.get("skipped"),
         "flags": flags, "env": env_extra or {}, "level_wanted": level, "kernels": rec.get("kernels")}
    VMS["arms"][name] = a          # rev 2g: a crash in the post-processing keeps the measured windows
    if ok:
        # rev 2b (agent 3 F1): one deadline for the whole arm: its cap, and never into the 2 x TP2 reserve
        t_end = min(time.perf_counter() + cap, HARD_STOP - VMS_RESERVE - 120)
        a["deadline_in_sec"] = round(t_end - time.perf_counter())
        _dp = "--data-parallel-size" in [str(x) for x in flags]
        a["diagnostic_only"] = bool(_dp)       # rev 2g (GPT R2F-01): no XL / per-rank canary on DP: never adoptable
        try:
            if gate in ("ref", "oracle"):
                rows, aa = vms_aa(name, deadline=t_end)
                VMS_ROWS[name] = rows
                a["aa"] = aa
                a["canary"] = vms_canary(level, deadline=t_end)
                if gate == "ref":
                    VMS_REF.update(rows)
                    a["gate"] = "reference" if aa["complete"] else "reference (A/A incomplete: every later gate INCOMPLETE)"
                elif aa["degenerate"] == 0 and aa["complete"] and a["canary"].get("foreign_leaks") == 0 \
                        and a["canary"].get("errors") == 0 and ((not _X) or (isinstance(aa.get("xl"), dict)
                        and aa["xl"].get("expected", 0) > 0 and aa["xl"].get("compared") == aa["xl"].get("expected")
                        and not aa["xl"].get("missing_margins"))):      # rev 2g (GPT R2F-10): XL A/A + canary clean
                    # rev 2: the oracle replaces the gate reference; the production reference is judged from its own rows
                    _r = VMS_ROWS.get("ref_fp8_mtp3_s16")
                    VMS_REF.clear()
                    VMS_REF.update(rows)
                    VMS["oracle_ready"] = True       # rev 2e (GPT audits HIGH-1 / E3): certification needs this
                    VMS["gate_reference"] = name + " (FP8, no speculation, prefix caching off)"
                    a["gate"] = "oracle"
                    if _r:
                        _v, _d = vms_judge(_r["serial_a"], _r["conc_a"], rows)
                        # rev 2g-b (agent 2 V1-d): the reference's warm (prefix-hit) A/A rows judged too; the worse counts
                        _v2, _d2 = vms_judge(_r["serial_b"], _r["conc_b"], rows)
                        _rank = {"PASS": 0, "INCOMPLETE": 1, "FAIL": 2}
                        _d["warm_verdict"], _d["warm_gate_class"] = _v2, _d2.get("gate_class")
                        if _rank.get(_v2, 2) > _rank.get(_v, 2):
                            _v, _d = _v2, dict(_d2, cold_verdict=_v, warm_verdict=_v2)
                        elif _v == "PASS" and _d2.get("gate_class") == "BOUNDARY":
                            _d["gate_class"] = "BOUNDARY"
                        VMS["ref_vs_oracle"] = {"verdict": _v, "detail": _d, "xl": vms_judge_xl(_r.get("xl_a"), rows)}
                        if VMS["arms"].get("ref_fp8_mtp3_s16"):
                            VMS["arms"]["ref_fp8_mtp3_s16"].update(gate=_v, gate_class=_d.get("gate_class"),
                                                                  xl_gate=(VMS["ref_vs_oracle"]["xl"] or {}).get("verdict"))
                else:
                    a["gate"] = "oracle unusable (incomplete or degenerate A/A, or a canary leak)"
            elif VMS_REF:
                s, c = vms_greedy(_S[:6], deadline=t_end), vms_greedy(_S, concurrent=True, deadline=t_end)
                xg = vms_greedy(_X[:4], concurrent=True, deadline=t_end) if (_X and not _dp) else None
                vms_save_rows(name + "_gate", {"serial": s, "conc": c, "xl": xg})
                a["gate"], a["gate_detail"] = vms_judge(s, c, VMS_REF, gate=bool(gate))
                a["gate_class"] = a["gate_detail"].get("gate_class")
                a["xl_gate"] = (vms_judge_xl(xg, VMS_REF) or {}).get("verdict") if xg else None
                a["canary"] = vms_canary(level, deadline=t_end)
            # rev 2: a FAIL is recorded and never adopted, but the speed is still measured
            a["short"] = vms_window(_S, level, _ts(75), name + "_short", deadline=t_end)
            # the next band runs unless the server is dead or dirty (a contaminated window would follow)
            _stop_next = lambda w: (w or {}).get("error") == "DIRTY_SERVER_AFTER_WINDOW" or (w or {}).get("idle_after") is False
            if time.perf_counter() < t_end and not _vms_dead() and not _stop_next(a["short"]):
                a["long"] = vms_window(_L, level, _ts(75), name + "_long", deadline=t_end)
            # rev 2d: the XL band (~85k prompts + up to 8k generated, ~100k per stream), at the streams the KV holds
            if _X and not _dp and time.perf_counter() < t_end and not _vms_dead() and not _stop_next(a.get("long")):
                _lx = max(1, min(level, (rec.get("kv_tokens") or 0) // 100_000)) if rec.get("kv_tokens") else min(level, 6)
                a["xl"] = vms_window(_X, _lx, _ts(75), name + "_xl", deadline=t_end)
            a["server_alive_after"] = not _vms_dead()
        except BaseException as _e:
            a["error"] = f"{type(_e).__name__}: {_e}"[:500]
        sig = VMS_SERVER.log_text() if VMS_SERVER is not None else ""
        # rev 2g-c (agent 3 A5): no whole-log 'Preempt' scan (the stats line carries it; one preemption in the diagnostic
        # XL window blocked adoption); the short / long windows require 0 preemptions instead (vms_window_healthy)
        a["log_errors"] = {k: sig.count(k) for k in ("Traceback", "out of memory", "OutOfMemory") if k in sig}
        a["preempt_log_lines"] = sig.count("Preempt")
    if VMS_SERVER is not None and "VLLM_ALLREDUCE_USE_FLASHINFER_PCIE_IPC" in (env_extra or {}):
        _lt = VMS_SERVER.log_text()        # rev 2d: did the PCIe-IPC all-reduce really engage?
        a["pcie_ipc"] = {"initialized": "Initialized FlashInfer PCIe IPC all-reduce" in _lt,
                         "not_provided": "does not provide PcieIpcAllReduceWorkspace" in _lt,
                         "other_warning": [l[-200:] for l in _lt.splitlines() if "PCIe IPC" in l and "Initialized" not in l][:4]}
    for band in ("short", "long", "xl"):
        w = a.get(band) or {}
        a[band + "_tok_s"], a["accept_" + band], a["level_" + band] = w.get("tok_s"), w.get("accept"), w.get("measured_level")
        a[band + "_e2e_tok_s"] = w.get("end_to_end_tok_s")
    vms_classify(a)
    VMS["arms"][name] = a
    VMS["order"].append(name)
    e14_save()
    print("VMS arm", name, json.dumps({k: v for k, v in a.items() if k not in ("flags", "short", "long", "xl")}, default=str)[:700], flush=True)
    return a


if VMS_ON and VMS_INST and not VMS["install"].get("error"):
    _ask = "Find the transformation rule of this ARC-AGI-2 task and describe it, then give the test output grids."
    _rows = lambda g: "\n".join(" ".join(str(v) for v in r) for r in g)
    _tt = lambda t: "".join(f"\nExample {i + 1}\nInput:\n{_rows(p['input'])}\nOutput:\n{_rows(p['output'])}\n"
                            for i, p in enumerate(t["train"])) + \
        "".join(f"\nTest input {j + 1}:\n{_rows(q['input'])}\n" for j, q in enumerate(t["test"]))
    _ids = lambda msgs: tokenizer.encode(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                                                        enable_thinking=True, reasoning_effort=REASONING_EFFORT),
                                         add_special_tokens=False)
    _S = [_ids([{"role": "user", "content": _ask + "\n" + _tt(CHALLENGES[t])}]) for t in TASK_IDS]
    _all = _load_json(ARC_ROOT / "arc-agi_evaluation_challenges.json")
    _bg, _n = [], 0
    for _t in sorted(_all):
        if _t in CHALLENGES:
            continue
        _bg.append(_tt(_all[_t]))
        _n += len(tokenizer.encode(_bg[-1], add_special_tokens=False))
        if _n >= int(os.environ.get("E14_LONG_TOKENS", 36000)):
            break
    _block = "Reference tasks (not to be solved):\n" + "\n".join(_bg)
    _L = [_ids([{"role": "system", "content": _block}, {"role": "user", "content": _ask + "\n" + _tt(CHALLENGES[t])}])
          for t in TASK_IDS]
    # rev 2d: the XL band, a shared ~85k-token block (vLLM max_model_len 106,496: 85k + 8k generated fits)
    _bgx, _nx = list(_bg), _n
    for _t in sorted(_all):
        if _nx >= int(os.environ.get("E14_XL_TOKENS", 85000)):
            break
        if _t in CHALLENGES or _tt(_all[_t]) in _bgx:
            continue
        _bgx.append(_tt(_all[_t]))
        _nx += len(tokenizer.encode(_bgx[-1], add_special_tokens=False))
    _blockx = "Reference tasks (not to be solved):\n" + "\n".join(_bgx)
    _X = [_ids([{"role": "system", "content": _blockx}, {"role": "user", "content": _ask + "\n" + _tt(CHALLENGES[t])}])
          for t in TASK_IDS]
    VMS["prompts"] = {"short_p50": statistics.median(len(x) for x in _S), "long_p50": statistics.median(len(x) for x in _L),
                      "xl_p50": statistics.median(len(x) for x in _X)}
    E14_STATE.enter("VMS_ARMS")
    vms_arm("ref_fp8_mtp3_s16", vms_flags(MODEL_PATH), gate="ref")
    VMS_RESERVE = 0.0                     # E22: no 2 x TP2 slot
    # rev 2: the oracle first (FP8, no speculation, prefix caching off), then the arms gated against it
    vms_arm("fp8_target_only_noprefix", vms_flags(MODEL_PATH, spec=0, prefix=False), gate="oracle")
    VMS.setdefault("gate_reference", "ref_fp8_mtp3_s16 (the oracle A/A was not usable)")
    _INT4_OK = vms_int4_mtp_static(VMS_INT4)
    try:          # rev 2g-c (checklist 9): the exact packages and prompts of this run
        VMS["pip_freeze"] = subprocess.run([(VMS.get("install") or {}).get("launch", [sys.executable])[0], "-m", "pip", "freeze"],
                                           capture_output=True, text=True, timeout=180).stdout[-20000:]
    except Exception as _e:
        VMS["pip_freeze"] = repr(_e)[:200]
    VMS["prompt_sha"] = {k_: [__import__("hashlib").sha256(json.dumps(list(x_)).encode()).hexdigest()[:16] for x_ in v_]
                         for k_, v_ in (("short", _S), ("long", _L), ("xl", _X or []))}
    _FULL = {"--compilation-config": json.dumps({"cudagraph_mode": "FULL_AND_PIECEWISE"})}
    _FDEC = {"--compilation-config": json.dumps({"cudagraph_mode": "FULL_DECODE_ONLY"})}
    _LARG = {"--speculative-config": json.dumps({"method": "mtp", "num_speculative_tokens": 3,
                                                 "use_local_argmax_reduction": True})}      # rev 2: dossier V32-ARGMAX
    _PMU = {"--prefix-match-unit": 64}                                                       # rev 2: dossier V32-HYBRID-REUSE
    _PMUS = {"--prefix-match-unit": 64, "--enable-mamba-shared-prefix-checkpoint": None}
    _N2 = {"NCCL_MIN_CTAS": "2", "NCCL_MAX_CTAS": "2"}
    _N4 = {"NCCL_MIN_CTAS": "4", "NCCL_MAX_CTAS": "4", "NCCL_NTHREADS": "512"}
    _N8 = {"NCCL_MIN_CTAS": "8", "NCCL_MAX_CTAS": "8", "NCCL_NTHREADS": "512"}
    _RING = {"NCCL_ALGO": "Ring"}

    def _spec_swap(flags, new_json):
        """the arm's --speculative-config replaced (vms_flags adds the default MTP 3 JSON; a second flag would be ambiguous)"""
        f = list(flags)
        i = f.index("--speculative-config")
        f[i + 1] = new_json
        return f

    # FP8 arms, gated against the FP8 target-only oracle
    PLAN = [
        # E22 (speed research 2026-10-29 + Resources dossiers v5): one lever per arm
        ("fp8_s16_triton_full", vms_flags(MODEL_PATH, backend="TRITON_ATTN", extra=_FULL), {}),
        ("fp8_s16_ssm_bf16", vms_flags(MODEL_PATH, extra={"--mamba-ssm-cache-dtype": "bfloat16"}), {}),
        ("fp8_s16_local_argmax", _spec_swap(vms_flags(MODEL_PATH), _LARG["--speculative-config"]), {}),
        # rev 2b (priority plan: FP16 vs BF16 GDN state in the same run): the fp16 control of fp8_s16_ssm_bf16
        ("fp8_s16_ssm_fp16", vms_flags(MODEL_PATH, extra={"--mamba-ssm-cache-dtype": "float16"}), {}),
    ]
    # INT4 arms, gated against the INT4 target-only oracle (rev 2: dossier P0-05 V32-INT4-ORACLE; run after the FP8 block)
    PLAN_INT4 = [
        ("int4_s20", vms_flags(VMS_INT4, seqs=20), dict(level=20)),
        # rev 2b (priority plan: MTP OFF / K1 / K3 at 16 / 20 active): speculation depth vs concurrency, prefix caching on
        ("int4_s16", vms_flags(VMS_INT4, seqs=16), dict(level=16)),
        ("int4_s16_mtp0", vms_flags(VMS_INT4, seqs=16, spec=0), dict(level=16)),
        ("int4_s20_mtp0", vms_flags(VMS_INT4, seqs=20, spec=0), dict(level=20)),
        ("int4_s16_mtp1", vms_flags(VMS_INT4, seqs=16, spec=1), dict(level=16)),
        ("int4_s20_mtp1", vms_flags(VMS_INT4, seqs=20, spec=1), dict(level=20)),
        ("int4_s20_local_argmax", _spec_swap(vms_flags(VMS_INT4, seqs=20), _LARG["--speculative-config"]), dict(level=20)),
        ("int4_s20_triton_full", vms_flags(VMS_INT4, seqs=20, backend="TRITON_ATTN", extra=_FULL), dict(level=20)),
        ("int4_s20_triton_fdec", vms_flags(VMS_INT4, seqs=20, backend="TRITON_ATTN", extra=_FDEC), dict(level=20)),
        ("int4_s20_nccl2", vms_flags(VMS_INT4, seqs=20), dict(level=20, env_extra=_N2)),
        ("int4_s20_nccl4", vms_flags(VMS_INT4, seqs=20), dict(level=20, env_extra=_N4)),
        ("int4_s20_nccl8", vms_flags(VMS_INT4, seqs=20), dict(level=20, env_extra=_N8)),
        ("int4_s20_ring", vms_flags(VMS_INT4, seqs=20), dict(level=20, env_extra=_RING)),
        ("int4_s20_pmu64", vms_flags(VMS_INT4, seqs=20, extra=_PMU), dict(level=20)),
        ("int4_s20_pmu64_sharedckpt", vms_flags(VMS_INT4, seqs=20, extra=_PMUS), dict(level=20)),
        ("int4_s20_pmu16_sharedckpt", vms_flags(VMS_INT4, seqs=20, extra={"--prefix-match-unit": 16,
                                                                          "--enable-mamba-shared-prefix-checkpoint": None}),
         dict(level=20)),
        ("int4_s24", vms_flags(VMS_INT4, seqs=24), dict(level=24)),
        ("int4_s20_b8192", vms_flags(VMS_INT4, seqs=20, batched=8192), dict(level=20)),
        ("int4_s24_triton_full_nccl8", vms_flags(VMS_INT4, seqs=24, backend="TRITON_ATTN", extra=_FULL),
         dict(level=24, env_extra=_N8)),
    ]
    VMS["dropped"] = {"fp8_mtp3_marlin (VLLM_TEST_FORCE_FP8_MARLIN=1)": "the env var is not in vLLM 0.32.0 envs.py "
                      "(removed by PR #52182): the arm would have run the reference unchanged; replaced by --linear-backend",
                      "fp8_mtp3_lb_marlin (--linear-backend marlin)": "on SM 8.9 the automatic FP8 block-128 kernel is "
                      "already MarlinFP8ScaledMM (kernels/linear/__init__.py:478-489; CUTLASS block FP8 needs SM 90+, "
                      "cutlass_scaled_mm_entry.cu:161-174): the arm = the reference"}
    for _name, _fl, _kw in PLAN:
        if _name.startswith("int4_") and not _INT4_OK:
            VMS["arms"][_name] = {"ok": False, "skipped": "INT4 MTP static check failed: " + VMS["int4_mtp_static"].get("why", "")}
            VMS["order"].append(_name)
            continue
        vms_arm(_name, _fl, **_kw)
    if _INT4_OK:
        VMS_REF.clear()
        vms_arm("int4_target_only_noprefix", vms_flags(VMS_INT4, seqs=20, spec=0, prefix=False), gate="ref", level=20)
        VMS["int4_gate_reference"] = "int4_target_only_noprefix (RedHat INT4, no speculation, prefix caching off)"
        # rev 2b (priority plan step 1): the vLLM step profile. INT4 MTP 3 at 20 streams on the 40k band; torch profiler for
        # ~6 s of steady decode (POST /start_profile, /stop_profile), then per-category CUDA kernel time of rank 0's trace
        # (NCCL all-reduce / all-gather, Marlin GEMM, GDN, attention, sampling, other) per decode step.
        _safe_profile("int4_s20_profile", vms_flags(VMS_INT4, seqs=20), 20)
    for _name, _fl, _kw in PLAN_INT4:
        if not _INT4_OK:
            VMS["arms"][_name] = {"ok": False, "skipped": "INT4 MTP static check failed: " + VMS["int4_mtp_static"].get("why", "")}
            VMS["order"].append(_name)
            continue
        vms_arm(_name, _fl, **_kw)
    VMS_RESERVE = 0.0
    for _a in VMS["arms"].values():       # rev 2e: the final classification (the reference is judged after the oracle)
        if "level_wanted" in _a:
            vms_classify(_a)
    E14_STATE.enter("VMS_DONE")
    e14_save()
else:
    VMS["skipped"] = "rerun" if RERUN else ("install failed" if VMS_INST is None or (VMS["install"] or {}).get("error") else "VMS off")
    print("VMS skipped:", VMS["skipped"], flush=True)


In [ ]:
# ==================================================================== CELL V3 — report, bundle
try:
    E14_STATE.enter("REPORT")
    _ref = VMS["arms"].get("ref_fp8_mtp3_s16") or {}
    _rs, _rl = _ref.get("short_tok_s") or 0, _ref.get("long_tok_s") or 0
    f = lambda v, d=0: "—" if v is None else (f"{v:.{d}f}" if isinstance(v, (int, float)) else str(v))
    xr = lambda v, r: "—" if not v or not r else f"{v / r:.2f}"
    L = ["# VMS: vLLM 0.32.0 engine-config matrix", "", f"profile {PROFILE}, rerun {RERUN}, install {VMS.get('install')}",
         f"prompts {VMS.get('prompts')}",
         # rev 2 (GPT pre-GPU audit): the oracle, the production reference vs it, the INT4 MTP static check, dropped arms
         f"gate reference: {VMS.get('gate_reference')} | oracle_ready (certification possible): {bool(VMS.get('oracle_ready'))}",
         "tok/s = steady decode after the prefill settled; e2e = output tokens from submission to the window end / wall (a "
         "submission-to-window-end rate, not completed-request task throughput); "
         "BOUNDARY = PASS with only near-tie divergences (adoptable under the documented tolerance, not exact: confirm "
         "on ARC tasks before production); certified_exact = adoptable and PASS_EXACT",
         f"reference (MTP 3 + prefix caching) vs the oracle: {(VMS.get('ref_vs_oracle') or {}).get('verdict')} "
         f"{json.dumps((VMS.get('ref_vs_oracle') or {}).get('detail'))[:400]}",
         f"INT4 MTP static check: {json.dumps(VMS.get('int4_mtp_static'))[:500]}",
         f"dropped arms: {json.dumps(VMS.get('dropped') or {})}",
         "note: the 40k band shares a ~36k system block; prefix-cache arms reuse it, the oracle and fp8_mtp3_noprefix "
         "prefill it again per request: tok/s is measured after the prefill settled (decode only), the recompute shows "
         "in the e2e column", "",
         "| arm | ok | gate | streams (8k / 40k) | 8k tok/s | x ref | 40k tok/s | x ref | accept 8k / 40k | KV tokens | KV max | preemptions | degenerate 8k / 40k | log errors | XL tok/s (streams) | x ref | accept XL | PCIe-IPC initialized | gate class / XL gate | canary leaks | e2e tok/s 8k / 40k | prefix hits / queries 40k | measurement / capacity / correctness | adoptable |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    _rx = _ref.get("xl_tok_s") or 0
    for k in VMS.get("order") or []:
        a = VMS["arms"][k]
        L.append(f"| {k} | {a.get('ok')} | {a.get('gate')} | {f(a.get('level_short'))} / {f(a.get('level_long'))} | "
                 f"{f(a.get('short_tok_s'))} | {xr(a.get('short_tok_s'), _rs)} | {f(a.get('long_tok_s'))} | {xr(a.get('long_tok_s'), _rl)} | "
                 f"{f(a.get('accept_short'), 2)} / {f(a.get('accept_long'), 2)} | {f(a.get('kv_tokens'))} | "
                 f"{f(((a.get('short') or {}).get('kv_max')), 2)} / {f(((a.get('long') or {}).get('kv_max')), 2)} | "
                 f"{f((a.get('short') or {}).get('preemptions'))} / {f((a.get('long') or {}).get('preemptions'))} | "
                 f"{f((a.get('short') or {}).get('degenerate'))} / {f((a.get('long') or {}).get('degenerate'))} | "
                 f"{a.get('log_errors') or a.get('skipped') or a.get('error') or (a.get('short') or {}).get('error') or (a.get('long') or {}).get('error') or ''} | "
                 f"{f(a.get('xl_tok_s'))} ({f(a.get('level_xl'))}){' ' + str((a.get('xl') or {}).get('error')) if (a.get('xl') or {}).get('error') else ''} | "
                 f"{xr(a.get('xl_tok_s'), _rx)} | {f(a.get('accept_xl'), 2)} | {(a.get('pcie_ipc') or {}).get('initialized', '')} | "
                 f"{a.get('gate_class') or ''} / {a.get('xl_gate') or ''} | {(a.get('canary') or {}).get('foreign_leaks', '')} | "
                 f"{f(a.get('short_e2e_tok_s'))} / {f(a.get('long_e2e_tok_s'))} | "
                 f"{f((a.get('long') or {}).get('prefix_cache_hits'))} / {f((a.get('long') or {}).get('prefix_cache_queries'))} | "
                 f"{a.get('measurement_ok')} / {a.get('capacity_ok')} / {a.get('correctness_ok')} | {a.get('adoptable')}{' certified_exact' if a.get('certified_exact') else ''} |")
    L += ["", "selected kernels per arm (from the server log):"] + [
        f"- {k}: {(VMS['arms'][k].get('kernels') or [])[:6]}" for k in VMS.get("order") or []]
    (VMS_DIR / "vms_report.md").write_text("\n".join(L) + "\n", encoding="utf-8")
    print("\n".join(L), flush=True)
    E14_STATE.end("done")
    e14_save()
    _pack = WORKING / "Q27X_VMS_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in sorted(VMS_DIR.rglob("*")):
            if fp.is_file() and fp.stat().st_size < 300e6:
                z.write(fp, arcname="vms/" + fp.relative_to(VMS_DIR).as_posix())
        for fp in sorted(E14_OUT.rglob("*")):
            if fp.is_file() and fp.stat().st_size < 300e6:
                z.write(fp, arcname="e14/" + fp.relative_to(E14_OUT).as_posix())
        for fp in (WORKING / "e14_state.json", WORKING / "e14_heartbeat.log"):
            if fp.exists():
                z.write(fp, arcname=fp.name)
    print("VMS bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
except BaseException:
    print("VMS report error:", traceback.format_exc()[-2000:], flush=True)
try:
    if VMS_SERVER is not None:
        VMS_SERVER.kill()
except Exception:
    pass
